# ResNet50 Fine-tuning on CIFAR-10

**流程概覽**
1. 安裝套件
2. 全域設定 & 隨機種子
3. 資料前處理（Resize + Scale）& 資料集分割（8:2）
4. 建立 DataLoader
5. 建立 ResNet50 模型
6. 訓練 / 驗證輔助函式（含早停）
7. **實驗 A** — 無增強 Fine-tune
8. **實驗 B** — Data Augmentation Fine-tune（旋轉、平移、翻轉）
9. 繪製兩實驗 Loss 曲線比較
10. 比較驗證集準確率，選出勝出策略
11. 完整重新訓練（訓練 + 驗證合併）
12. 測試集評估：Accuracy / Precision / F1 / 混淆矩陣
13. 預測結果視覺化（3 張原圖）


## 0｜安裝套件

In [2]:
# 若尚未安裝，取消下行註解執行
# !pip install torch torchvision scikit-learn seaborn matplotlib


## 1｜Import & 全域設定

In [3]:
import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (
    accuracy_score, precision_score, f1_score, confusion_matrix
)

# ── 全域超參數 ──
SEED         = 42
NUM_CLASSES  = 10
EPOCHS       = 20
BATCH_SIZE   = 32
LR           = 1e-4
PATIENCE     = 3          # 早停耐心值
VAL_RATIO    = 0.2        # 驗證集比例
DATA_DIR     = "./data"
SAVE_DIR     = "./checkpoints"
CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

os.makedirs(SAVE_DIR, exist_ok=True)

# ── 固定隨機種子 ──
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── 設備 ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用設備：{device}")
if device.type == "cuda":
    print(f"GPU：{torch.cuda.get_device_name(0)}")


使用設備：cuda
GPU：NVIDIA GeForce RTX 4060 Laptop GPU


## 2｜資料前處理 Transform 定義

- **前處理**：`Resize(224×224)` → `ToTensor`（自動將像素值從 [0, 255] 縮放至 [0.0, 1.0]）
- **增強**：在前處理基礎上加入 隨機水平翻轉、隨機旋轉 ±15°、隨機平移 10%


In [4]:
# 基礎 Transform（驗證集 / 測試集 / 實驗 A 訓練集）
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),      
])

# 增強 Transform（實驗 B 訓練集）
aug_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),                    # 隨機水平翻轉
    transforms.RandomRotation(degrees=15),                     # 隨機旋轉 ±15°
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # 隨機平移 10%
    transforms.ToTensor(),      
])

print("Transform 定義完成")


Transform 定義完成


## 3｜載入 CIFAR-10 & 訓練集資料集分割（8:2）

In [5]:
print("載入 CIFAR-10 ...")

full_train_base = datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True, transform=base_transform
)
full_train_aug = datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True, transform=aug_transform
)
test_dataset = datasets.CIFAR10(
    root=DATA_DIR, train=False, download=True, transform=base_transform
)

# 分割索引
n_total = len(full_train_base)   # 50,000
n_val   = int(n_total * VAL_RATIO)
n_train = n_total - n_val

indices   = list(range(n_total))
random.shuffle(indices)
train_idx = indices[:n_train]
val_idx   = indices[n_train:]

print(f"訓練集：{n_train} 張 | 驗證集：{n_val} 張 | 測試集：{len(test_dataset)} 張")

# Subset
train_subset_base = Subset(full_train_base, train_idx)
train_subset_aug  = Subset(full_train_aug,  train_idx)
val_subset        = Subset(full_train_base, val_idx)


載入 CIFAR-10 ...


100%|██████████| 170M/170M [00:13<00:00, 13.1MB/s] 


訓練集：40000 張 | 驗證集：10000 張 | 測試集：10000 張


## 4｜建立 DataLoader

In [6]:
train_loader_base = DataLoader(
    train_subset_base, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)
train_loader_aug = DataLoader(
    train_subset_aug,  batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)

print("DataLoader 建立完成")


DataLoader 建立完成


## 5｜建立 ResNet50 模型

載入 ImageNet 預訓練權重，將最後的 FC 層替換為 `Dropout(0.3) → Linear(2048, 10)`。


In [7]:
def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model.to(device)

# 快速確認模型最後幾層
sample_model = build_model()
print(sample_model.fc)
del sample_model


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\jay02/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:02<00:00, 38.8MB/s]


Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=2048, out_features=10, bias=True)
)


## 6｜訓練 / 驗證輔助函式

In [8]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += images.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += images.size(0)
    return total_loss / total, correct / total


def run_experiment(name, train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE):
    """執行完整訓練實驗（早停監控 Val Loss），回傳 model / history / best_val_acc / stopped_epoch。"""
    print(f"\n{'='*55}")
    print(f"  實驗：{name}")
    print(f"{'='*55}")

    model     = build_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float("inf")   # 早停監控 Val Loss
    best_val_acc  = 0.0            # 記錄對應的最佳驗證準確率
    best_state    = None
    patience_cnt  = 0
    stopped_epoch = epochs         # 記錄實際停止的 epoch 數

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        print(f"  Epoch {epoch:2d}/{epochs} | "
              f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f} | "
              f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

        # 早停監控 Val Loss（越小越好）
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_val_acc  = vl_acc
            best_state    = copy.deepcopy(model.state_dict())
            patience_cnt  = 0
            stopped_epoch = epoch
            torch.save(best_state, os.path.join(SAVE_DIR, f"{name}_best.pth"))
        else:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"  [早停] 連續 {patience} 個 epoch Val Loss 未下降，停止訓練。")
                break

    model.load_state_dict(best_state)
    print(f"\n  {name} 最佳 Val Loss：{best_val_loss:.4f}  對應 Val Acc：{best_val_acc:.4f}  (Epoch {stopped_epoch})")
    return {
        "model":         model,
        "history":       history,
        "best_val_acc":  best_val_acc,
        "best_val_loss": best_val_loss,
        "stopped_epoch": stopped_epoch,
        "best_state":    best_state,
    }


print("函式定義完成")


## 7｜實驗 A — 無增強 Fine-tune

In [9]:
result_a = run_experiment(
    name="ExpA_NoAug",
    train_loader=train_loader_base,
    val_loader=val_loader,
)



  實驗：ExpA_NoAug
  Epoch  1/20 | Train Loss: 0.3760  Acc: 0.8804 | Val Loss: 0.1658  Acc: 0.9440


KeyboardInterrupt: 

## 8｜實驗 B — Data Augmentation Fine-tune

In [ ]:
result_b = run_experiment(
    name="ExpB_Aug",
    train_loader=train_loader_aug,
    val_loader=val_loader,
)


## 9｜繪製兩實驗 Loss 曲線比較

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training & Validation Loss Curves", fontsize=15, fontweight="bold")

for ax, result, title in zip(
    axes,
    [result_a, result_b],
    ["Experiment A — No Augmentation", "Experiment B — With Augmentation"]
):
    history = result["history"]
    ep = range(1, len(history["train_loss"]) + 1)
    ax.plot(ep, history["train_loss"], label="Train Loss", color="#e05c5c", linewidth=2)
    ax.plot(ep, history["val_loss"],   label="Val Loss",   color="#5c8ee0", linewidth=2, linestyle="--")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Loss 曲線已儲存為 loss_curves.png")


## 10｜比較驗證集準確率，選出勝出策略

In [ ]:
print("="*55)
print("  驗證集準確率比較")
print(f"  Exp A（無增強）：Acc={result_a['best_val_acc']:.4f}  Val Loss={result_a['best_val_loss']:.4f}  最佳 Epoch={result_a['stopped_epoch']}")
print(f"  Exp B（有增強）：Acc={result_b['best_val_acc']:.4f}  Val Loss={result_b['best_val_loss']:.4f}  最佳 Epoch={result_b['stopped_epoch']}")

use_aug       = result_b["best_val_acc"] >= result_a["best_val_acc"]
winner        = "Exp B（有增強）" if use_aug else "Exp A（無增強）"
best_result   = result_b if use_aug else result_a
final_epochs  = best_result["stopped_epoch"]   # 勝出方的最佳 epoch 數

print(f"  ✓ 勝出策略    ：{winner}")
print(f"  ✓ 完整訓練將跑：{final_epochs} 個 epoch（無早停）")
print("="*55)


## 11｜完整重新訓練（訓練 + 驗證合併）

In [ ]:
print("="*55)
print("  完整重新訓練（訓練 + 驗證合併）")
print(f"  策略：{'Data Augmentation' if use_aug else '無增強'}")
print(f"  訓練 epoch 數：{final_epochs}（依勝出實驗最佳 epoch，無早停）")
print("="*55)

full_dataset = full_train_aug if use_aug else full_train_base
full_loader  = DataLoader(
    full_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=4, pin_memory=True
)

final_model = build_model()
criterion   = nn.CrossEntropyLoss()
optimizer   = optim.Adam(final_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=final_epochs)

best_loss, best_state = float("inf"), None

for epoch in range(1, final_epochs + 1):
    tr_loss, tr_acc = train_one_epoch(final_model, full_loader, criterion, optimizer)
    scheduler.step()
    print(f"  Epoch {epoch:2d}/{final_epochs} | Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}")

    if tr_loss < best_loss:
        best_loss  = tr_loss
        best_state = copy.deepcopy(final_model.state_dict())
        torch.save(best_state, os.path.join(SAVE_DIR, "final_best.pth"))

final_model.load_state_dict(best_state)
print("\n完整重新訓練完成，模型已儲存至 checkpoints/final_best.pth")


## 12｜測試集評估：Accuracy / Precision / F1 / 混淆矩陣

In [ ]:
final_model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds  = final_model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
f1   = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
cm   = confusion_matrix(all_labels, all_preds)

print("="*55)
print("  測試集最終評估結果")
print("="*55)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  F1 Score  : {f1:.4f}")
print("="*55)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CIFAR10_CLASSES,
    yticklabels=CIFAR10_CLASSES,
    linewidths=0.4, ax=ax
)
ax.set_title("Confusion Matrix — Test Set", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Predicted Label", fontsize=11)
ax.set_ylabel("True Label",      fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("混淆矩陣已儲存為 confusion_matrix.png")


## 13｜預測結果視覺化（3 張原圖）

In [ ]:
def to_displayable(tensor: torch.Tensor) -> np.ndarray:
    """ToTensor 後的 [0,1] tensor → HWC numpy，可直接 imshow。"""
    img = tensor.numpy().transpose(1, 2, 0)  # C,H,W → H,W,C
    return np.clip(img, 0, 1)


num_samples = 3
indices  = random.sample(range(len(test_dataset)), num_samples)
images   = [test_dataset[i][0] for i in indices]
true_lbs = [test_dataset[i][1] for i in indices]

final_model.eval()
batch = torch.stack(images).to(device)
with torch.no_grad():
    probs = torch.softmax(final_model(batch), dim=1).cpu()

fig, axes = plt.subplots(1, num_samples, figsize=(5 * num_samples, 5))
fig.suptitle("Model Predictions on Test Samples", fontsize=14, fontweight="bold", y=1.02)

for ax, img_tensor, true_lb, prob in zip(axes, images, true_lbs, probs):
    ax.imshow(to_displayable(img_tensor))
    ax.axis("off")
    pred_cls   = prob.argmax().item()
    confidence = prob[pred_cls].item()
    color = "green" if pred_cls == true_lb else "red"
    ax.set_title(
        f"True:  {CIFAR10_CLASSES[true_lb]}\n"
        f"Pred:  {CIFAR10_CLASSES[pred_cls]}  ({confidence*100:.1f}%)",
        fontsize=11, color=color, fontweight="bold"
    )

plt.tight_layout()
plt.savefig("prediction_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print("預測圖片已儲存為 prediction_samples.png")


---
## 完成！

所有輸出檔案：
| 檔案 | 說明 |
|------|------|
| `loss_curves.png` | 兩實驗 Train/Val Loss 並排比較 |
| `confusion_matrix.png` | 10×10 混淆矩陣 Heatmap |
| `prediction_samples.png` | 3 張測試圖原圖 + 預測結果 |
| `checkpoints/ExpA_NoAug_best.pth` | 實驗 A 最佳權重 |
| `checkpoints/ExpB_Aug_best.pth` | 實驗 B 最佳權重 |
| `checkpoints/final_best.pth` | 最終完整訓練最佳權重 |
